# Lesson 15: Prompting, Planning, and Learning with Claude Code

**Time:** ~40 minutes | **Cost:** $0 (no API calls)

You know what Claude Code is (Lesson 13) and how to use it (Lesson 14). This lesson teaches the **skills** that make the difference between "Claude Code kinda works" and "Claude Code does exactly what I need."

Three skills:
1. **Prompting** — how to write instructions that get the right result
2. **Planning** — how to break down complex tasks before building
3. **Learning** — how to understand unfamiliar code and debug problems

## The Development Workflow

```
1. Understand → 2. Plan → 3. Implement → 4. Verify → 5. Iterate
```

This maps to what you already know:

| Step | What you do | Agent pipeline equivalent |
|------|------------|---------------------------|
| **Understand** | Read the code, understand the current state | Research Agent searches the web |
| **Plan** | Design what needs to change, in what order | Outline Agent creates structure |
| **Implement** | Claude Code writes the code | Content Writer writes the article |
| **Verify** | You review — does it match what you wanted? | Human review of generated content |
| **Iterate** | Fix what's wrong, improve what's close | Edit and regenerate |

**Key insight**: Steps 1, 2, 4, and 5 are YOUR job. Step 3 is Claude Code's job. You spend more time thinking and reviewing than coding.

## Skill 1: Prompting for Code Changes

The principles from Lesson 6 (Prompts and Context) apply directly. But code prompts need specific elements:

### The 4 elements of a good code prompt:

1. **WHERE** — which file(s) to modify
2. **WHAT** — what behavior to add or change
3. **HOW** — constraints, patterns to follow
4. **VERIFY** — how to confirm it works

### Examples from our project:

**Bad**: "Make the writer better"
**Good**: "In `output/backend/agents/content_writer.py`, add an instruction that articles must include at least 3 internal links to other articles. Use `list_all_articles()` to find existing article topics. After implementing, create a test article to verify links are included."

**Bad**: "Fix the storage"
**Good**: "In `output/backend/tools/storage.py`, the `save_article` function doesn't validate that `topic` is non-empty. Add a check at the start of the function that raises a ValueError if topic is empty or whitespace-only."

In [ ]:
# Let's analyze what makes prompts effective
# Rate each prompt on the 4 elements: WHERE, WHAT, HOW, VERIFY

prompts = [
    {
        "prompt": "Fix the agents",
        "where": False,  # No file specified
        "what": False,   # "Fix" what exactly?
        "how": False,    # No constraints
        "verify": False, # No way to check
        "score": 0,
    },
    {
        "prompt": "In content_writer.py, make articles longer",
        "where": True,   # File specified
        "what": False,   # "Longer" is vague
        "how": False,    # How much longer? What sections?
        "verify": False, # No verification method
        "score": 1,
    },
    {
        "prompt": "In output/backend/agents/content_writer.py, update instructions to require minimum 2000 words. Add a note that each H2 section should be at least 200 words. Run the writer on a test topic to verify.",
        "where": True,
        "what": True,    # Minimum 2000 words, 200 per section
        "how": True,     # Specific word counts
        "verify": True,  # Test on a topic
        "score": 4,
    },
]

for p in prompts:
    elements = sum([p["where"], p["what"], p["how"], p["verify"]])
    print(f"Score: {elements}/4")
    if len(p['prompt']) > 80:
        print(f"  Prompt: \"{p['prompt'][:80]}...\"")
    else:
        print(f"  Prompt: \"{p['prompt']}\"")
    missing = []
    if not p["where"]: missing.append("WHERE (no file)")
    if not p["what"]: missing.append("WHAT (vague goal)")
    if not p["how"]: missing.append("HOW (no constraints)")
    if not p["verify"]: missing.append("VERIFY (no test)")
    if missing:
        print(f"  Missing: {', '.join(missing)}")
    print()

## Skill 2: Planning Before Building

### When to use plan mode

Plan mode (`Shift+Tab` in Claude Code) tells Claude Code to **explore and design** before making changes. Use it when:

- You're making changes to **more than 2 files**
- You're **unsure** about the best approach
- The task is **complex** (new feature, architecture change)
- You want to **understand** the current code first

### How plan mode works

1. You describe what you want
2. Claude Code reads relevant files, explores the codebase
3. It presents a **plan** — which files to change, what changes to make, in what order
4. You **review** the plan — does it make sense? Does it follow existing patterns?
5. You approve → Claude Code implements

### Example: Adding a Proofreader agent

Without plan mode:
> "Add a proofreader agent"
> Claude Code might create a file that doesn't follow existing patterns, forget to register it in \_\_init\_\_.py, or miss the team.py integration.

With plan mode:
> "I want to add a Proofreader agent. Use plan mode to explore the existing agent files and design the approach first."
> Claude Code reads content_writer.py, image_finder.py, aio_analyzer.py, team.py, \_\_init\_\_.py, and presents:
> - Create `agents/proofreader.py` following the content_writer.py pattern
> - Add export to `agents/__init__.py`
> - Add as member in `agents/team.py`
> - Use Claude Sonnet (matching other agents)
> You review, approve, then it implements.

### Reviewing plans

When Claude Code shows a plan, check:
- Does it modify the **right files**? (not files you didn't expect)
- Does it follow **existing patterns**? (same model, same tool style)
- Is the **order correct**? (create before import, define before use)
- Is anything **missing**? (forgot to update \_\_init\_\_.py?)

## Skill 3: Learning Unfamiliar Concepts

The next lessons will show you production code with concepts you haven't learned yet: FastAPI, React, SSE streaming, CORS. **This is normal.** You don't need to know everything — you need to know how to ask.

### Strategy 1: "Explain this like I'm not a developer"

When you see code you don't understand:
> "Explain what this code does in `serve.py` lines 15-30. I'm not a developer — explain the PURPOSE, not the syntax."

Claude Code will explain the business logic, not the Python details.

### Strategy 2: "What would break if I changed this?"

Before modifying unfamiliar code:
> "I want to change the API route from `/api/articles` to `/api/posts`. What other files reference this path? What would break?"

Claude Code traces all dependencies.

### Strategy 3: "Show me a simpler version"

When code is too complex:
> "The `serve.py` file uses AgentOS which is complex. Can you show me a simplified version (10 lines max) that demonstrates what it does conceptually?"

### Strategy 4: "What does this error mean?"

When something breaks:
> "I got this error when running `python output/backend/serve.py`:
> ```
> ModuleNotFoundError: No module named 'agno.os'
> ```
> What does this mean and how do I fix it?"

**Don't try to understand errors yourself.** Paste the full error into Claude Code and ask what happened.

In [ ]:
# This is actual code from serve.py (simplified)
# Don't worry about understanding every line
# Focus on identifying WHAT it does, not HOW

production_code = '''
from agno.os import AgentOS

app = FastAPI()

@app.get("/api/articles")
def list_articles():
    return list_all_articles()

@app.post("/api/articles/{article_id}")  
def update_article(article_id: str, body: dict):
    return update_article_content(article_id, body["content"])

AgentOS(teams=[team], base_app=app).start(port=7777)
'''

# Practice: identify the 4 elements without understanding the syntax
print("What this code does (high level):")
print("1. Creates a web server (FastAPI)")
print("2. Adds 2 API routes for articles (list + update)")
print("3. Connects the agent team to the web server (AgentOS)")
print("4. Starts everything on port 7777")
print()
print("What you DON'T need to know:")
print("- What @app.get means (it's a decorator \u2014 Python concept)")
print("- How FastAPI routing works internally")
print("- What AgentOS does under the hood")
print()
print("What you DO need to know:")
print("- This file is the web backend entry point")
print("- It connects agents to a web interface")
print("- Port 7777 is where the server runs")
print("- Articles can be listed and updated through web requests")

## Debugging with Claude Code

Debugging follows a simple pattern:

1. **See the error** — copy the FULL error message (not just the last line)
2. **Ask Claude Code** — paste it and ask "What went wrong and how do I fix it?"
3. **Understand the fix** — ask "Why did this happen?" if you want to learn
4. **Verify** — run the code again to confirm it's fixed

### Common errors you'll encounter:

| Error | What it means | Your prompt to Claude Code |
|-------|--------------|----------------------------|
| `ModuleNotFoundError` | A package isn't installed | "I got ModuleNotFoundError for X. Install it and verify." |
| `ImportError` | Wrong import path | "Fix this ImportError in [file]. The module moved." |
| `KeyError` | Dictionary missing a key | "This KeyError in [file] says key 'X' is missing. Check where it's set." |
| `FileNotFoundError` | File path is wrong | "This file path in [file] is wrong. Find the correct path." |
| `ConnectionError` | API/server not reachable | "The server can't connect to [service]. Check if it's running." |

**The key skill**: Don't try to debug alone. The moment you see an error you don't understand, paste it into Claude Code. That's what it's for.

## Lesson 15 Summary

### Three skills for AI-assisted development:

| Skill | Key takeaway |
|-------|-------------|
| **Prompting** | Use the 4 elements: WHERE, WHAT, HOW, VERIFY |
| **Planning** | Use plan mode for complex tasks. Review plans before approving. |
| **Learning** | Ask Claude Code to explain unfamiliar code. Paste full errors. |

### The 5-step workflow:
```
Understand → Plan → Implement → Verify → Iterate
     YOU       YOU    CLAUDE CODE    YOU      YOU
```

Your value is in steps 1, 2, 4, and 5. Claude Code handles step 3.

**Next lesson:** Building the Content Writer — your first look at production agent code, through the lens of "how would you direct Claude Code to build this?"

## Exercise

Write Claude Code prompts for each scenario. Rate each prompt using the 4-element checklist (WHERE, WHAT, HOW, VERIFY).

In [ ]:
# Exercise: Write 5 prompts of increasing difficulty
# For each, fill in the blanks to make it a complete 4/4 prompt

# 1. EASY: Change the writer's tone
easy = """
In output/backend/agents/___,
change the writing tone from formal to ___.
Keep all other instructions about ___ and word count unchanged.
Create a test article to verify the ___ changed.
"""

# 2. MEDIUM: Add a field to storage
medium = """
In output/backend/tools/___,
add a ___ field to the article metadata in save_article.
Calculate it as word_count / ___ (average reading speed).
Update ___ to display this field when listing articles.
Test by saving an article and checking the new field appears.
"""

# 3. HARD: Add a new agent
hard = """
Use plan mode first. I want to add a ___ agent that checks
articles for grammar and spelling errors.
Follow the pattern in output/backend/agents/___.py.
Register it in output/backend/agents/___.
Add it as a member in output/backend/agents/___.
Use Claude ___ (same model as other agents).
Test by running it on an existing article.
"""

# 4. DEBUGGING: Fix an error
debug = """
When I run `python output/backend/serve.py`, I get:
```
ImportError: cannot import name '___' from 'agents'
```
Find where this import is used, check what's actually
exported from agents/__init__.py, and fix the ___.
"""

# 5. LEARNING: Understand unfamiliar code
learn = """
Explain what output/backend/___ does.
I'm not a developer \u2014 explain the PURPOSE of each section,
not the Python syntax. Focus on:
- What does this file do in the overall system?
- What would break if we deleted it?
- How does it connect to the ___ files?
"""

for name, prompt in [("Easy", easy), ("Medium", medium), ("Hard", hard), ("Debug", debug), ("Learn", learn)]:
    print(f"=== {name} ===")
    print(prompt)

<details>
<summary>Click to reveal answers</summary>

**1. Easy** (change the writer's tone):
- `content_writer.py`
- `casual and conversational`
- `SEO`
- `tone`

**2. Medium** (add a field to storage):
- `storage.py`
- `reading_time_minutes`
- `200`
- `list_all_articles`

**3. Hard** (add a new agent):
- `proofreader`
- `content_writer`
- `__init__.py`
- `team.py`
- `Sonnet`

**4. Debug** (fix an error):
- The import name that doesn't exist in `agents/__init__.py`
- `mismatch` between the import and what's actually exported

**5. Learn** (understand unfamiliar code):
- `serve.py`
- `agents/`

</details>